# DB2 process DB-COMP files — v3

This is one self-contained DB-COMP processing notebook.

It preserves the DB-COMP-specific manifest, document IDs, folders and canonical output locations from the earlier notebook, but replaces the duplicated/older extraction layer with the newer manual/antitrust standard:

- smart HTML/TXT encoding detection;
- scored HTML content-container selection;
- PyMuPDF sorted text, PyMuPDF sorted blocks and optional `pdftotext -layout`;
- quality comparison across native candidates;
- conservative MinerU fallback;
- a minimum MinerU score gain before OCR replaces usable native text;
- readable and regex/LLM text outputs;
- clean, candidate and failed-row manifests;
- extraction versioning and efficient reruns.

Canonical regex/LLM text remains under:

`data/processed/com_db_comp/text/`


## 1. Imports and configuration

In [16]:

from __future__ import annotations

import os
import re
import sys
import json
import shutil
import hashlib
import tempfile
import subprocess
import unicodedata
from collections import Counter
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple

import pandas as pd
from tqdm.auto import tqdm

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from bs4 import BeautifulSoup, UnicodeDammit
except Exception:
    BeautifulSoup = None
    UnicodeDammit = None

try:
    from charset_normalizer import from_bytes as charset_from_bytes
except Exception:
    charset_from_bytes = None

try:
    from ftfy import fix_text as ftfy_fix_text
except Exception:
    ftfy_fix_text = None

# ---------------------------------------------------------------------
# Project paths — preserve DB-COMP conventions
# ---------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "data").exists() and (candidate / "output").exists():
        PROJECT_ROOT = candidate
        break

# Override manually if required:
# PROJECT_ROOT = Path("/home/edik/projects/eccjeu").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "com_db_comp"
PROCESSED_DIR = DATA_DIR / "processed" / "com_db_comp"
TEXT_DIR = PROCESSED_DIR / "text"
MARKDOWN_DIR = PROCESSED_DIR / "markdown"
CANDIDATE_DATA_DIR = PROCESSED_DIR / "candidates"
ERROR_DIR = PROCESSED_DIR / "errors"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_db_comp"

DOWNLOAD_MANIFEST_PATH = OUTPUT_DIR / "db_comp_download_manifest.csv"
LEGACY_DOWNLOAD_MANIFEST_PATH = OUTPUT_DIR / "download_manifest.csv"

CLEAN_FILE_MANIFEST_PATH = OUTPUT_DIR / "db_comp_clean_file_manifest.csv"
CLEAN_FILE_MANIFEST_XLSX_PATH = OUTPUT_DIR / "db_comp_clean_file_manifest.xlsx"
CANDIDATE_MANIFEST_PATH = OUTPUT_DIR / "db_comp_extraction_candidate_manifest.csv"
FAILED_FILE_MANIFEST_PATH = OUTPUT_DIR / "db_comp_failed_cleaning_rows.csv"

for path in [
    TEXT_DIR,
    MARKDOWN_DIR,
    CANDIDATE_DATA_DIR,
    ERROR_DIR,
    OUTPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Production controls
# ---------------------------------------------------------------------
EXTRACTION_VERSION = 3
FORCE_REPROCESS = True
PROCESS_LIMIT = None
SAVE_EVERY = 25

# Candidate diagnostics always go into the candidate manifest.
# Physical raw/readable/regex files for every candidate are expensive.
SAVE_ALL_CANDIDATES = False

# ---------------------------------------------------------------------
# Native PDF and OCR behavior
# ---------------------------------------------------------------------
RUN_PDFTOTEXT = True
RUN_MINERU = True
RUN_MINERU_FOR_ALL_PDFS = False

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")

MINERU_TIMEOUT_SECONDS = 300
MIN_NATIVE_SCORE_TO_RUN_OCR = 35.0
MIN_MINERU_SCORE_GAIN = 2.5

# These may launch OCR. Mild language/token warnings do not.
SERIOUS_OCR_REASONS = {
    "very_short_text",
    "replacement_characters",
    "possible_mojibake",
    "private_use_characters",
    "repeated_character_garbage",
}

SERIOUS_REVIEW_REASONS = {
    "very_short_text",
    "replacement_characters",
    "possible_mojibake",
    "private_use_characters",
    "repeated_character_garbage",
}

if MINERU_CLI:
    MINERU_COMMAND_TEMPLATE = [
        MINERU_CLI,
        "-p", "{input_pdf}",
        "-o", "{output_dir}",
        "-m", "ocr",
        "-b", "pipeline",
    ]
else:
    MINERU_COMMAND_TEMPLATE = [
        "mineru",
        "-p", "{input_pdf}",
        "-o", "{output_dir}",
        "-m", "ocr",
        "-b", "pipeline",
    ]

# Quality thresholds
MIN_CHARS_ANY = 100
MIN_PDF_CHARS_GOOD = 1200
MIN_SCORE_ACCEPTABLE = 35.0
MIN_SCORE_OK = 55.0

print("Python:", sys.version)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Download manifest:", DOWNLOAD_MANIFEST_PATH)
print("Clean manifest:", CLEAN_FILE_MANIFEST_PATH)
print("Text output:", TEXT_DIR)
print("Readable output uses __readable.txt in the same directory.")
print("Markdown output:", MARKDOWN_DIR)
print("Candidate manifest:", CANDIDATE_MANIFEST_PATH)
print("PyMuPDF:", fitz is not None)
print("BeautifulSoup:", BeautifulSoup is not None)
print("charset-normalizer:", charset_from_bytes is not None)
print("ftfy:", ftfy_fix_text is not None)
print("pdftotext:", PDFTOTEXT_CLI)
print("MinerU:", MINERU_CLI)
print("FORCE_REPROCESS:", FORCE_REPROCESS)


Python: 3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]
PROJECT_ROOT: /home/edik/projects/eccjeu
Download manifest: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_download_manifest.csv
Clean manifest: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_clean_file_manifest.csv
Text output: /home/edik/projects/eccjeu/data/processed/com_db_comp/text
Readable output uses __readable.txt in the same directory.
Markdown output: /home/edik/projects/eccjeu/data/processed/com_db_comp/markdown
Candidate manifest: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_extraction_candidate_manifest.csv
PyMuPDF: True
BeautifulSoup: True
charset-normalizer: True
ftfy: True
pdftotext: /usr/bin/pdftotext
MinerU: /home/edik/projects/.venv/bin/mineru
FORCE_REPROCESS: True


## 2. Load and prepare the DB-COMP manifest

In [17]:
TEXT_EXTENSIONS = {'.txt', '.text'}
HTML_EXTENSIONS = {'.html', '.htm', '.xhtml'}
PDF_EXTENSIONS = {'.pdf'}
SUPPORTED_EXTENSIONS = TEXT_EXTENSIONS | HTML_EXTENSIONS | PDF_EXTENSIONS

PATH_COLUMNS = [
    'final_local_path', 'local_path', 'existing_local_path', 'intended_local_path',
    'raw_file_path', 'download_path', 'target_path', 'file_path', 'raw_path',
    'saved_path', 'proposed_path', 'renamed_path'
]
URL_COLUMNS = ['file_url', 'download_url', 'source_url', 'url', 'document_url', 'href']
CASE_COLUMNS = ['case_number', 'case_number_modified', 'modified_case_number', 'case_id', 'case_ref']
DOC_ID_COLUMNS = ['dbcomp_document_id', 'document_id', 'file_id', 'db_comp_document_id']
TITLE_COLUMNS = ['document_title', 'title', 'case_name', 'display_title', 'name']


def read_csv_safely(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, low_memory=False)


def first_existing_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    return next((c for c in candidates if c in df.columns), None)


def first_nonempty(row: pd.Series, cols: List[str]) -> str:
    for col in cols:
        if col in row.index:
            val = row.get(col)
            if pd.notna(val) and str(val).strip() and str(val).strip().lower() not in {'nan', 'none', 'null'}:
                return str(val).strip()
    return ''


def normalize_possible_path(value: str) -> Optional[Path]:
    if not value or str(value).strip().lower() in {'nan', 'none', 'null'}:
        return None
    raw = str(value).strip()
    p = Path(raw)
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([
            PROJECT_ROOT / p,
            DATA_DIR / p,
            RAW_DIR / p,
            OUTPUT_DIR / p,
        ])
    for c in candidates:
        if c.exists() and c.is_file():
            return c.resolve()
    return None


def infer_file_type(path: Path) -> str:
    ext = path.suffix.lower()
    if ext in PDF_EXTENSIONS:
        return 'pdf'
    if ext in HTML_EXTENSIONS:
        return 'html'
    if ext in TEXT_EXTENSIONS:
        return 'text'
    return ext.lstrip('.') or 'unknown'


def safe_id(value: str) -> str:
    value = str(value or '').strip()
    value = re.sub(r'[^A-Za-z0-9_.-]+', '_', value)
    value = re.sub(r'_+', '_', value).strip('_.-')
    return value or 'missing_id'


def load_download_manifest() -> pd.DataFrame:
    if DOWNLOAD_MANIFEST_PATH.exists():
        path = DOWNLOAD_MANIFEST_PATH
    elif LEGACY_DOWNLOAD_MANIFEST_PATH.exists():
        path = LEGACY_DOWNLOAD_MANIFEST_PATH
        print(f'Using legacy fallback manifest: {path}')
    else:
        raise FileNotFoundError(
            f'Could not find {DOWNLOAD_MANIFEST_PATH} or {LEGACY_DOWNLOAD_MANIFEST_PATH}'
        )
    df = read_csv_safely(path)
    df['__manifest_path'] = str(path)
    return df


def prepare_download_rows(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for idx, row in df.iterrows():
        raw_path = normalize_possible_path(first_nonempty(row, PATH_COLUMNS))
        if raw_path is None:
            rows.append({
                **row.to_dict(),
                'raw_file_path': '',
                'raw_file_type': '',
                'processing_eligible': False,
                'processing_skip_reason': 'missing_or_unresolved_file_path',
            })
            continue
        if raw_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            rows.append({
                **row.to_dict(),
                'raw_file_path': str(raw_path),
                'raw_file_type': infer_file_type(raw_path),
                'processing_eligible': False,
                'processing_skip_reason': 'unsupported_file_type',
            })
            continue
        rows.append({
            **row.to_dict(),
            'raw_file_path': str(raw_path),
            'raw_file_type': infer_file_type(raw_path),
            'processing_eligible': True,
            'processing_skip_reason': '',
            'source_url_for_processing': first_nonempty(row, URL_COLUMNS),
            'title_for_processing': first_nonempty(row, TITLE_COLUMNS),
        })
    out = pd.DataFrame(rows)

    # Normalize preferred ID columns while preserving originals.
    if 'dbcomp_document_id' not in out.columns:
        doc_col = first_existing_col(out, DOC_ID_COLUMNS)
        out['dbcomp_document_id'] = out[doc_col].astype(str).str.strip() if doc_col else ''
    if 'case_number' not in out.columns:
        case_col = first_existing_col(out, CASE_COLUMNS)
        out['case_number'] = out[case_col].astype(str).str.strip() if case_col else ''

    # Deterministic fallback if any IDs are still missing.
    missing = out['dbcomp_document_id'].isna() | out['dbcomp_document_id'].astype(str).str.strip().isin(['', 'nan', 'None', 'null'])
    if missing.any():
        fallback_order = out.loc[missing].sort_values(['source_url_for_processing', 'raw_file_path'], na_position='last').index
        for n, i in enumerate(fallback_order, start=9000):
            out.at[i, 'dbcomp_document_id'] = str(n)

    out['dbcomp_document_id'] = out['dbcomp_document_id'].astype(str).str.strip()
    out['case_number'] = out['case_number'].astype(str).replace({'nan': ''}).str.strip()
    return out


download_df = load_download_manifest()
process_df = prepare_download_rows(download_df)

print('Rows in download manifest:', len(download_df))
print('Eligible files:', int(process_df['processing_eligible'].sum()))
print('Ineligible files:', int((~process_df['processing_eligible']).sum()))
display(process_df['raw_file_type'].value_counts(dropna=False))
display(process_df[['dbcomp_document_id', 'case_number', 'raw_file_path', 'raw_file_type', 'processing_eligible', 'processing_skip_reason']].head(20))


# Candidate engine uses a common work-manifest interface.
work_manifest = process_df.copy()
work_manifest["document_id"] = work_manifest["dbcomp_document_id"].astype(str).str.strip()
work_manifest["file_format"] = work_manifest["raw_file_type"].replace({"text": "txt"})
work_manifest["raw_file_exists"] = work_manifest["raw_file_path"].apply(
    lambda value: Path(str(value)).exists() if str(value).strip() else False
)
work_manifest["cleaning_target"] = (
    work_manifest["processing_eligible"].fillna(False)
    & work_manifest["raw_file_exists"]
)

if PROCESS_LIMIT is not None:
    work_manifest = work_manifest.head(PROCESS_LIMIT).copy()

print("Work-manifest rows:", len(work_manifest))
print("Cleaning targets:", int(work_manifest["cleaning_target"].sum()))


Rows in download manifest: 833
Eligible files: 833
Ineligible files: 0


raw_file_type
pdf    833
Name: count, dtype: int64

,dbcomp_document_id,case_number,raw_file_path,raw_file_type,processing_eligible,processing_skip_reason
0,1061,93,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
1,1064,399904,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
2,1065,40481,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
3,1066,40360,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
4,1067,40291,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
5,1068,40208,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
6,1069,40169,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
7,1072,40113,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
8,1073,40105,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,
9,1074,40098,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,True,


Work-manifest rows: 833
Cleaning targets: 833


## 3. Smart decoding, cleaning and quality diagnostics

In [18]:

MOJIBAKE_PATTERNS = [
    "Ã", "Â", "â€™", "â€œ", "â€", "ðŸ", "�",
]

LEGAL_TERMS = {
    "en": ["commission", "decision", "court", "article", "applicant", "undertaking", "competition"],
    "de": ["kommission", "entscheidung", "gericht", "artikel", "kläger", "unternehmen", "wettbewerb"],
    "fr": ["commission", "décision", "cour", "article", "requérant", "entreprise", "concurrence"],
    "it": ["commissione", "decisione", "corte", "articolo", "ricorrente", "impresa", "concorrenza"],
}

COMMON_WORDS = {
    "en": ["the", "of", "and", "to", "in", "that", "for", "on", "with"],
    "de": ["der", "die", "das", "und", "von", "zu", "in", "für", "mit"],
    "fr": ["de", "la", "le", "et", "des", "les", "du", "pour", "dans"],
    "it": ["di", "la", "il", "e", "del", "della", "per", "in", "con"],
}


def read_file_bytes(path: Path) -> bytes:
    return path.read_bytes()


def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def score_decoded_text(text: str) -> float:
    if not text:
        return -100.0
    n = len(text)
    replacement_ratio = text.count("�") / max(n, 1)
    control_ratio = sum(
        1 for ch in text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)
    mojibake_ratio = sum(text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in text) / max(n, 1)
    return (
        printable_ratio * 30
        - replacement_ratio * 300
        - control_ratio * 200
        - mojibake_ratio * 180
    )


def extract_declared_html_encoding(data: bytes) -> Optional[str]:
    head = data[:8192]
    ascii_head = head.decode("ascii", errors="ignore")
    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, ascii_head, flags=re.I)
        if match:
            return match.group(1).strip()
    return None


def decode_bytes_smart(data: bytes, is_html: bool = False) -> Tuple[str, Dict[str, Any]]:
    candidates: List[Tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(("utf-8-sig", "bom", data.decode("utf-8-sig", errors="replace")))
    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(("utf-16", "bom", data.decode("utf-16", errors="replace")))

    declared = extract_declared_html_encoding(data) if is_html else None
    if declared:
        try:
            candidates.append((declared, "declared", data.decode(declared, errors="replace")))
        except Exception:
            pass

    if is_html and UnicodeDammit is not None:
        try:
            dammit = UnicodeDammit(data, is_html=True, smart_quotes_to=None)
            if dammit.unicode_markup:
                candidates.append((
                    dammit.original_encoding or "unknown",
                    "unicode_dammit",
                    dammit.unicode_markup,
                ))
        except Exception:
            pass

    try:
        candidates.append(("utf-8", "strict_utf8", data.decode("utf-8", errors="strict")))
    except UnicodeDecodeError:
        pass

    if charset_from_bytes is not None:
        try:
            matches = charset_from_bytes(data)
            for match in list(matches)[:4]:
                text = str(match)
                candidates.append((
                    getattr(match, "encoding", None) or "unknown",
                    "charset_normalizer",
                    text,
                ))
        except Exception:
            pass

    for encoding in ["cp1252", "latin-1"]:
        try:
            candidates.append((encoding, "fallback", data.decode(encoding, errors="replace")))
        except Exception:
            pass

    if not candidates:
        candidates.append(("utf-8", "last_resort", data.decode("utf-8", errors="replace")))

    unique = {}
    for enc, source, text in candidates:
        key = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()
        unique.setdefault(key, (enc, source, text))

    scored = []
    for enc, source, text in unique.values():
        base_score = score_decoded_text(text)
        repaired = None
        repaired_score = None

        if ftfy_fix_text is not None:
            try:
                repaired = ftfy_fix_text(text)
                repaired_score = score_decoded_text(repaired)
            except Exception:
                repaired = None

        use_repaired = (
            repaired is not None
            and repaired != text
            and repaired_score is not None
            and repaired_score > base_score + 0.5
        )
        final_text = repaired if use_repaired else text
        final_score = repaired_score if use_repaired else base_score

        scored.append({
            "encoding": enc,
            "encoding_source": source,
            "text": final_text,
            "decode_score": float(final_score),
            "ftfy_applied": bool(use_repaired),
            "declared_encoding": declared or "",
        })

    best = max(scored, key=lambda x: x["decode_score"])
    meta = {k: v for k, v in best.items() if k != "text"}
    meta["decode_candidate_count"] = len(scored)
    return normalize_unicode(best["text"]), meta


def clean_text_readable(text: str) -> str:
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def clean_text_for_regex_and_llm(text: str) -> str:
    text = clean_text_readable(text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def repeated_character_ratio(text: str) -> float:
    if not text:
        return 1.0
    repeated = sum(len(m.group(0)) for m in re.finditer(r"(.)\1{5,}", text, flags=re.S))
    return repeated / max(len(text), 1)


def duplicate_line_ratio(text: str) -> float:
    lines = [
        re.sub(r"\s+", " ", line).strip().lower()
        for line in text.splitlines()
        if len(re.sub(r"\s+", " ", line).strip()) >= 20
    ]
    if not lines:
        return 0.0
    counts = Counter(lines)
    duplicate_instances = sum(count - 1 for count in counts.values() if count > 1)
    return duplicate_instances / max(len(lines), 1)


def private_use_ratio(text: str) -> float:
    if not text:
        return 0.0
    return sum(
        0xE000 <= ord(ch) <= 0xF8FF
        for ch in text
    ) / max(len(text), 1)


def token_diagnostics(text: str) -> Dict[str, float]:
    tokens = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text, flags=re.UNICODE)
    if not tokens:
        return {
            "one_char_token_ratio": 1.0,
            "very_long_token_ratio": 1.0,
            "no_vowel_token_ratio": 1.0,
            "mixed_alnum_token_ratio": 0.0,
        }

    vowels = set("aeiouyäöüàâæçéèêëîïôœùûüÿ")
    one_char = sum(len(t) == 1 for t in tokens)
    very_long = sum(len(t) > 30 for t in tokens)
    no_vowel = sum(
        len(t) >= 5 and not any(ch.lower() in vowels for ch in t)
        for t in tokens
    )
    mixed = sum(any(ch.isalpha() for ch in t) and any(ch.isdigit() for ch in t) for t in tokens)

    n = len(tokens)
    return {
        "one_char_token_ratio": one_char / n,
        "very_long_token_ratio": very_long / n,
        "no_vowel_token_ratio": no_vowel / n,
        "mixed_alnum_token_ratio": mixed / n,
    }


def language_plausibility(text: str) -> Tuple[float, str]:
    words = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text.lower(), flags=re.UNICODE)
    if not words:
        return 0.0, ""

    counts = Counter(words)
    scores = {}
    for lang in COMMON_WORDS:
        common_hits = sum(counts[w] for w in COMMON_WORDS[lang])
        legal_hits = sum(counts[w] for w in LEGAL_TERMS[lang])
        scores[lang] = min(1.0, (common_hits * 0.5 + legal_hits * 2.0) / max(len(words) * 0.01, 1))

    best_lang = max(scores, key=scores.get)
    return float(scores[best_lang]), best_lang


def text_quality_profile(raw_text: str, clean_text: str, meta: Dict[str, Any], file_format: str) -> Dict[str, Any]:
    raw_text = raw_text or ""
    clean_text = clean_text or ""
    n = len(clean_text)

    alpha_ratio = sum(ch.isalpha() for ch in clean_text) / max(n, 1)
    digit_ratio = sum(ch.isdigit() for ch in clean_text) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in clean_text) / max(n, 1)
    replacement_ratio = clean_text.count("�") / max(n, 1)
    mojibake_ratio = sum(clean_text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    control_ratio = sum(
        1 for ch in clean_text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)

    token_stats = token_diagnostics(clean_text)
    lang_score, detected_language = language_plausibility(clean_text)
    repeat_ratio = repeated_character_ratio(clean_text)
    duplicate_ratio = duplicate_line_ratio(clean_text)
    pua_ratio = private_use_ratio(clean_text)

    length_component = min(25.0, n / 800.0)
    score = (
        length_component
        + printable_ratio * 15
        + min(alpha_ratio / 0.65, 1.0) * 10
        + lang_score * 20
        - replacement_ratio * 250
        - mojibake_ratio * 180
        - control_ratio * 250
        - pua_ratio * 250
        - repeat_ratio * 100
        - duplicate_ratio * 20
        - token_stats["one_char_token_ratio"] * 18
        - token_stats["very_long_token_ratio"] * 80
        - token_stats["no_vowel_token_ratio"] * 18
    )

    reasons = []
    if n < MIN_CHARS_ANY:
        reasons.append("very_short_text")
    elif file_format == "pdf" and n < MIN_PDF_CHARS_GOOD:
        reasons.append("short_pdf_text")
    if replacement_ratio > 0.002:
        reasons.append("replacement_characters")
    if mojibake_ratio > 0.0005:
        reasons.append("possible_mojibake")
    if pua_ratio > 0.0005:
        reasons.append("private_use_characters")
    if repeat_ratio > 0.01:
        reasons.append("repeated_character_garbage")
    if duplicate_ratio > 0.20:
        reasons.append("many_duplicate_lines")
    if token_stats["one_char_token_ratio"] > 0.30:
        reasons.append("many_one_character_tokens")
    if token_stats["very_long_token_ratio"] > 0.01:
        reasons.append("many_very_long_tokens")
    if lang_score < 0.10 and n > 1000:
        reasons.append("low_language_plausibility")

    if n < MIN_CHARS_ANY or score < MIN_SCORE_ACCEPTABLE:
        quality_flag = "failed"
    elif reasons:
        quality_flag = "fishy"
    else:
        quality_flag = "ok"

    return {
        "quality_score": round(float(score), 3),
        "quality_flag": quality_flag,
        "quality_reasons": "|".join(reasons),
        "n_chars_raw": len(raw_text),
        "n_chars_clean": n,
        "n_pages": meta.get("n_pages"),
        "alpha_ratio": round(alpha_ratio, 6),
        "digit_ratio": round(digit_ratio, 6),
        "printable_ratio": round(printable_ratio, 6),
        "replacement_ratio": round(replacement_ratio, 8),
        "mojibake_ratio": round(mojibake_ratio, 8),
        "private_use_ratio": round(pua_ratio, 8),
        "repeated_character_ratio": round(repeat_ratio, 8),
        "duplicate_line_ratio": round(duplicate_ratio, 8),
        "language_score": round(lang_score, 6),
        "detected_language": detected_language,
        **{k: round(v, 8) for k, v in token_stats.items()},
    }


## 4. HTML, TXT, native PDF and MinerU candidates

In [19]:

HTML_REMOVE_SELECTORS = [
    "script", "style", "noscript", "nav", "footer", "header", "form", "aside",
    "[class*='cookie']", "[id*='cookie']", "[class*='share']",
    "[class*='language']", "[aria-label*='language']",
    "[class*='pagination']", "[class*='print']",
]

HTML_CANDIDATE_SELECTORS = [
    "main", "article", "#document1", "#TexteOnly", "#text",
    ".document-content", ".document", ".content", "body",
]


def html_container_score(tag) -> float:
    text = tag.get_text(" ", strip=True)
    if not text:
        return -100.0

    links = tag.find_all("a")
    link_text_chars = sum(len(a.get_text(" ", strip=True)) for a in links)
    link_density = link_text_chars / max(len(text), 1)
    paragraphs = len(tag.find_all(["p", "div", "li", "table"]))
    lang_score, _ = language_plausibility(text)

    return (
        min(len(text), 150_000) / 1500
        + min(paragraphs, 200) * 0.08
        + lang_score * 15
        - link_density * 40
    )


def extract_text_from_html(path: Path) -> Tuple[str, Dict[str, Any]]:
    data = read_file_bytes(path)
    html, decode_meta = decode_bytes_smart(data, is_html=True)

    if BeautifulSoup is None:
        text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.I | re.S)
        text = re.sub(r"<[^>]+>", " ", text)
        return normalize_unicode(text), {
            "method": "html_regex",
            "n_pages": None,
            **decode_meta,
        }

    soup = BeautifulSoup(html, "html.parser")

    for selector in HTML_REMOVE_SELECTORS:
        try:
            for tag in soup.select(selector):
                tag.decompose()
        except Exception:
            pass

    candidates = []
    for selector in HTML_CANDIDATE_SELECTORS:
        try:
            for tag in soup.select(selector):
                text = tag.get_text("\n", strip=True)
                if len(text) >= 100:
                    candidates.append((html_container_score(tag), selector, text))
        except Exception:
            pass

    if candidates:
        _, selected_selector, text = max(candidates, key=lambda x: x[0])
    else:
        selected_selector = "document"
        text = soup.get_text("\n", strip=True)

    return normalize_unicode(text), {
        "method": "html_bs4_best_container",
        "n_pages": None,
        "html_selected_container": selected_selector,
        "html_candidate_count": len(candidates),
        **decode_meta,
    }


def extract_text_from_plain(path: Path) -> Tuple[str, Dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(read_file_bytes(path), is_html=False)
    return text, {"method": "plain_text_smart_decode", "n_pages": None, **decode_meta}


def extract_pdf_pymupdf_text(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            text = page.get_text("text", sort=True) or ""
            chunks.append(f"\n\n[page {page_no}]\n{text}")
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_text_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pymupdf_blocks(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            blocks = page.get_text("blocks", sort=True) or []
            block_texts = []
            for block in blocks:
                if len(block) >= 5:
                    text = str(block[4] or "").strip()
                    if text:
                        block_texts.append(text)
            chunks.append(f"\n\n[page {page_no}]\n" + "\n\n".join(block_texts))
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_blocks_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pdftotext(path: Path) -> Tuple[str, Dict[str, Any]]:
    if not PDFTOTEXT_CLI:
        raise RuntimeError("pdftotext is not installed or not on PATH.")

    with tempfile.TemporaryDirectory(prefix="pdftotext_") as tmpdir:
        out_txt = Path(tmpdir) / "output.txt"
        cmd = [PDFTOTEXT_CLI, "-layout", "-enc", "UTF-8", str(path), str(out_txt)]
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if proc.returncode != 0:
            raise RuntimeError(proc.stderr[-2000:] or "pdftotext failed")
        text = out_txt.read_text(encoding="utf-8", errors="replace")

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_pdftotext_layout",
        "n_pages": n_pages,
    }


def format_mineru_command(input_pdf: Path, output_dir: Path) -> List[str]:
    return [
        part.format(input_pdf=str(input_pdf), output_dir=str(output_dir))
        for part in MINERU_COMMAND_TEMPLATE
    ]


def run_mineru(input_pdf: Path, output_dir: Path) -> Dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = format_mineru_command(input_pdf, output_dir)

    try:
        env = os.environ.copy()
        env.setdefault("ORT_LOG_SEVERITY_LEVEL", "3")
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=MINERU_TIMEOUT_SECONDS,
            env=env,
        )
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "cmd": " ".join(cmd),
            "stdout_tail": proc.stdout[-2000:],
            "stderr_tail": proc.stderr[-2000:],
        }
    except Exception as exc:
        return {
            "ok": False,
            "returncode": None,
            "cmd": " ".join(cmd),
            "error": repr(exc),
        }


def find_mineru_text(output_dir: Path) -> Optional[Path]:
    candidates = []
    for pattern in ["**/*.md", "**/*.txt"]:
        candidates.extend(output_dir.glob(pattern))
    candidates = [p for p in candidates if p.is_file() and p.stat().st_size > 0]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_size)


def extract_pdf_mineru(path: Path) -> Tuple[str, Dict[str, Any]]:
    with tempfile.TemporaryDirectory(prefix="mineru_") as tmpdir:
        out_dir = Path(tmpdir)
        result = run_mineru(path, out_dir)
        if not result.get("ok"):
            raise RuntimeError(result.get("error") or result.get("stderr_tail") or "MinerU failed")

        text_file = find_mineru_text(out_dir)
        if text_file is None:
            raise RuntimeError("MinerU completed but produced no non-empty markdown/text file.")

        text = text_file.read_text(encoding="utf-8", errors="replace")

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_mineru",
        "n_pages": n_pages,
    }


def make_candidate(
    document_id: str,
    method: str,
    raw_text: str,
    meta: Dict[str, Any],
    file_format: str,
) -> Dict[str, Any]:
    readable_text = clean_text_readable(raw_text)
    regex_text = clean_text_for_regex_and_llm(raw_text)
    quality = text_quality_profile(raw_text, regex_text, meta, file_format)

    return {
        "document_id": document_id,
        "candidate_method": method,
        "raw_text": raw_text,
        "readable_text": readable_text,
        "clean_text": regex_text,
        "meta": meta,
        **quality,
    }



def generate_candidates(
    row: Dict[str, Any],
) -> Tuple[List[Dict[str, Any]], List[str]]:
    path = Path(str(row["raw_file_path"]))
    file_format = str(row.get("file_format", "")).lower()
    document_id = str(row["document_id"])
    candidates: List[Dict[str, Any]] = []
    errors: List[str] = []

    def attempt(method_name, extractor):
        try:
            raw_text, meta = extractor(path)
            candidates.append(
                make_candidate(
                    document_id=document_id,
                    method=method_name,
                    raw_text=raw_text,
                    meta=meta,
                    file_format=file_format,
                )
            )
        except Exception as exc:
            errors.append(f"{method_name}: {repr(exc)}")

    if file_format == "html":
        attempt("html_best_container", extract_text_from_html)

    elif file_format in {"txt", "text"}:
        attempt("plain_text_smart_decode", extract_text_from_plain)

    elif file_format == "pdf":
        attempt(
            "pdf_pymupdf_text_sorted",
            extract_pdf_pymupdf_text,
        )
        attempt(
            "pdf_pymupdf_blocks_sorted",
            extract_pdf_pymupdf_blocks,
        )

        if RUN_PDFTOTEXT and PDFTOTEXT_CLI:
            attempt(
                "pdf_pdftotext_layout",
                extract_pdf_pdftotext,
            )

        native_candidates = [
            candidate
            for candidate in candidates
            if candidate.get("candidate_method") != "pdf_mineru"
        ]
        best_native = max(
            native_candidates,
            key=candidate_preference,
            default=None,
        )

        best_native_score = (
            float(best_native.get("quality_score", -999))
            if best_native
            else -999
        )
        best_native_reasons = {
            reason
            for reason in str(
                best_native.get("quality_reasons", "")
                if best_native
                else ""
            ).split("|")
            if reason
        }

        should_run_ocr = (
            RUN_MINERU
            and MINERU_CLI is not None
            and (
                RUN_MINERU_FOR_ALL_PDFS
                or best_native is None
                or best_native.get("quality_flag") == "failed"
                or best_native_score < MIN_NATIVE_SCORE_TO_RUN_OCR
                or bool(best_native_reasons & SERIOUS_OCR_REASONS)
            )
        )

        if should_run_ocr:
            attempt("pdf_mineru", extract_pdf_mineru)

    else:
        errors.append(
            f"unsupported_file_format: {file_format}"
        )

    return candidates, errors


## 5. Candidate selection and output helpers

In [20]:

def candidate_preference(
    candidate: Dict[str, Any],
) -> Tuple[float, int]:
    # Native methods win ties because they are less destructive than OCR.
    method_order = {
        "html_best_container": 6,
        "plain_text_smart_decode": 6,
        "pdf_pymupdf_text_sorted": 5,
        "pdf_pdftotext_layout": 4,
        "pdf_pymupdf_blocks_sorted": 3,
        "pdf_mineru": 1,
    }
    return (
        float(candidate.get("quality_score", -999)),
        method_order.get(
            candidate.get("candidate_method", ""),
            0,
        ),
    )


def select_best_candidate(
    candidates: List[Dict[str, Any]],
) -> Tuple[Optional[Dict[str, Any]], str]:
    if not candidates:
        return None, "no_candidates"

    native = [
        candidate
        for candidate in candidates
        if candidate.get("candidate_method") != "pdf_mineru"
    ]
    mineru = next(
        (
            candidate
            for candidate in candidates
            if candidate.get("candidate_method") == "pdf_mineru"
        ),
        None,
    )

    best_native = max(
        native,
        key=candidate_preference,
        default=None,
    )

    if mineru is None:
        selected = max(
            candidates,
            key=candidate_preference,
        )
        return selected, "best_available_candidate"

    if best_native is None:
        return mineru, "native_missing_mineru_available"

    native_score = float(
        best_native.get("quality_score", -999)
    )
    mineru_score = float(
        mineru.get("quality_score", -999)
    )

    # Complete native failure is the exception to the gain rule.
    if (
        best_native.get("quality_flag") == "failed"
        and mineru.get("quality_flag") != "failed"
    ):
        return mineru, "native_failed_mineru_valid"

    gain = mineru_score - native_score
    if gain >= MIN_MINERU_SCORE_GAIN:
        return (
            mineru,
            f"mineru_gain_{gain:.3f}_meets_threshold",
        )

    return (
        best_native,
        f"mineru_gain_{gain:.3f}_below_threshold",
    )


def save_candidate_files(
    document_id: str,
    candidate: Dict[str, Any],
) -> Dict[str, str]:
    method = safe_id(
        candidate["candidate_method"]
    )
    folder = CANDIDATE_DATA_DIR / safe_id(document_id)
    folder.mkdir(parents=True, exist_ok=True)

    raw_path = folder / f"{method}__raw.txt"
    readable_path = folder / f"{method}__readable.txt"
    regex_path = folder / f"{method}__regex.txt"

    raw_path.write_text(
        candidate.get("raw_text", ""),
        encoding="utf-8",
    )
    readable_path.write_text(
        candidate.get("readable_text", ""),
        encoding="utf-8",
    )
    regex_path.write_text(
        candidate.get("clean_text", ""),
        encoding="utf-8",
    )

    return {
        "candidate_raw_path": str(raw_path),
        "candidate_readable_path": str(readable_path),
        "candidate_regex_path": str(regex_path),
    }


def output_paths_for(
    document_id: str,
) -> Tuple[Path, Path, Path]:
    safe_document_id = safe_id(document_id)
    clean_path = TEXT_DIR / f"{safe_document_id}.txt"
    readable_path = (
        TEXT_DIR / f"{safe_document_id}__readable.txt"
    )
    markdown_path = (
        MARKDOWN_DIR / f"{safe_document_id}.md"
    )
    return clean_path, readable_path, markdown_path


def write_markdown(
    row: Dict[str, Any],
    clean_text: str,
    markdown_path: Path,
    selected: Dict[str, Any],
) -> None:
    meta = {
        "document_id": row.get("document_id"),
        "source": "com_db_comp",
        "dbcomp_document_id": row.get(
            "dbcomp_document_id"
        ),
        "case_number": row.get("case_number"),
        "title": row.get("title_for_processing"),
        "source_url": row.get(
            "source_url_for_processing"
        ),
        "file_format": row.get("file_format"),
        "raw_file_path": row.get("raw_file_path"),
        "extraction_version": EXTRACTION_VERSION,
        "selected_method": selected.get(
            "candidate_method"
        ),
        "quality_score": selected.get(
            "quality_score"
        ),
        "quality_flag": selected.get(
            "quality_flag"
        ),
        "quality_reasons": selected.get(
            "quality_reasons"
        ),
        "created_at": datetime.now().isoformat(
            timespec="seconds"
        ),
    }

    header = (
        "---\n"
        + "\n".join(
            f"{key}: {json.dumps(value, ensure_ascii=False)}"
            for key, value in meta.items()
        )
        + "\n---\n\n"
    )
    markdown_path.write_text(
        header + clean_text,
        encoding="utf-8",
    )


def load_existing_manifests():
    old_clean = pd.DataFrame()
    old_candidates = pd.DataFrame()

    if (
        CLEAN_FILE_MANIFEST_PATH.exists()
        and CLEAN_FILE_MANIFEST_PATH.stat().st_size > 0
    ):
        try:
            old_clean = pd.read_csv(
                CLEAN_FILE_MANIFEST_PATH,
                low_memory=False,
            )
        except pd.errors.EmptyDataError:
            old_clean = pd.DataFrame()

    if (
        CANDIDATE_MANIFEST_PATH.exists()
        and CANDIDATE_MANIFEST_PATH.stat().st_size > 0
    ):
        try:
            old_candidates = pd.read_csv(
                CANDIDATE_MANIFEST_PATH,
                low_memory=False,
            )
        except pd.errors.EmptyDataError:
            old_candidates = pd.DataFrame()

    clean_lookup = {}
    if not old_clean.empty and "document_id" in old_clean:
        for _, row in old_clean.iterrows():
            clean_lookup[str(row["document_id"])] = (
                row.to_dict()
            )

    candidate_lookup = {}
    if (
        not old_candidates.empty
        and "document_id" in old_candidates
    ):
        for document_id, group in old_candidates.groupby(
            old_candidates["document_id"].astype(str)
        ):
            candidate_lookup[str(document_id)] = (
                group.to_dict("records")
            )

    return (
        old_clean,
        old_candidates,
        clean_lookup,
        candidate_lookup,
    )


(
    previous_clean_manifest,
    previous_candidate_manifest,
    previous_clean_lookup,
    previous_candidate_lookup,
) = load_existing_manifests()


def existing_output_is_current(
    document_id: str,
) -> bool:
    if FORCE_REPROCESS:
        return False

    row = previous_clean_lookup.get(
        str(document_id)
    )
    if not row:
        return False

    try:
        version_ok = (
            int(float(row.get("extraction_version", 0)))
            == EXTRACTION_VERSION
        )
    except Exception:
        version_ok = False

    required = [
        row.get("clean_text_path", ""),
        row.get("readable_text_path", ""),
        row.get("markdown_path", ""),
    ]
    paths_ok = all(
        Path(str(path)).exists()
        and Path(str(path)).stat().st_size > 0
        for path in required
        if str(path).strip()
    )

    return version_ok and paths_ok


## 6. Process files and write manifests

In [21]:

def process_one(
    row: Dict[str, Any],
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    document_id = str(row["document_id"])
    (
        clean_text_path,
        readable_text_path,
        markdown_path,
    ) = output_paths_for(document_id)

    base = {
        **row,
        "processed_at": datetime.now().isoformat(
            timespec="seconds"
        ),
        "extraction_version": EXTRACTION_VERSION,
        "clean_text_path": str(clean_text_path),
        "readable_text_path": str(readable_text_path),
        "markdown_path": str(markdown_path),
    }

    if not bool(row.get("cleaning_target", False)):
        return {
            **base,
            "clean_success": False,
            "quality_flag": "skipped",
            "quality_score": None,
            "needs_manual_review": True,
            "candidate_count": 0,
            "error": row.get(
                "processing_skip_reason",
                "not_a_cleaning_target",
            ),
        }, []

    if existing_output_is_current(document_id):
        reused = dict(
            previous_clean_lookup[str(document_id)]
        )
        reused["processed_at"] = datetime.now().isoformat(
            timespec="seconds"
        )
        reused["selection_reason"] = (
            "reused_current_v3_output"
        )
        return (
            reused,
            previous_candidate_lookup.get(
                str(document_id),
                [],
            ),
        )

    candidates, errors = generate_candidates(row)

    candidate_rows = []
    for candidate in candidates:
        saved_paths = {}
        if SAVE_ALL_CANDIDATES:
            saved_paths = save_candidate_files(
                document_id,
                candidate,
            )

        candidate_rows.append({
            "document_id": document_id,
            "dbcomp_document_id": row.get(
                "dbcomp_document_id"
            ),
            "case_number": row.get("case_number"),
            "raw_file_path": row.get("raw_file_path"),
            "file_format": row.get("file_format"),
            "candidate_method": candidate.get(
                "candidate_method"
            ),
            "quality_score": candidate.get(
                "quality_score"
            ),
            "quality_flag": candidate.get(
                "quality_flag"
            ),
            "quality_reasons": candidate.get(
                "quality_reasons"
            ),
            "n_chars_raw": candidate.get(
                "n_chars_raw"
            ),
            "n_chars_clean": candidate.get(
                "n_chars_clean"
            ),
            "n_pages": candidate.get("n_pages"),
            "detected_language": candidate.get(
                "detected_language"
            ),
            "language_score": candidate.get(
                "language_score"
            ),
            "replacement_ratio": candidate.get(
                "replacement_ratio"
            ),
            "mojibake_ratio": candidate.get(
                "mojibake_ratio"
            ),
            "private_use_ratio": candidate.get(
                "private_use_ratio"
            ),
            "repeated_character_ratio": candidate.get(
                "repeated_character_ratio"
            ),
            "duplicate_line_ratio": candidate.get(
                "duplicate_line_ratio"
            ),
            "one_char_token_ratio": candidate.get(
                "one_char_token_ratio"
            ),
            "very_long_token_ratio": candidate.get(
                "very_long_token_ratio"
            ),
            "no_vowel_token_ratio": candidate.get(
                "no_vowel_token_ratio"
            ),
            "meta_json": json.dumps(
                candidate.get("meta", {}),
                ensure_ascii=False,
            ),
            **saved_paths,
        })

    selected, selection_reason = (
        select_best_candidate(candidates)
    )

    if selected is None:
        return {
            **base,
            "clean_success": False,
            "quality_flag": "failed",
            "quality_score": None,
            "needs_manual_review": True,
            "candidate_count": 0,
            "selection_reason": selection_reason,
            "candidate_errors": " | ".join(errors),
            "error": "no_candidate_succeeded",
        }, candidate_rows

    clean_text = selected.get("clean_text", "")
    readable_text = selected.get(
        "readable_text",
        "",
    )

    clean_text_path.write_text(
        clean_text,
        encoding="utf-8",
    )
    readable_text_path.write_text(
        readable_text,
        encoding="utf-8",
    )
    write_markdown(
        row,
        clean_text,
        markdown_path,
        selected,
    )

    ranked = sorted(
        candidates,
        key=candidate_preference,
        reverse=True,
    )
    score_listing = "; ".join(
        f"{candidate['candidate_method']}="
        f"{candidate['quality_score']:.3f}"
        for candidate in ranked
    )

    selected_reasons = {
        reason
        for reason in str(
            selected.get("quality_reasons", "")
        ).split("|")
        if reason
    }
    needs_manual_review = bool(
        selected_reasons & SERIOUS_REVIEW_REASONS
    ) or selected.get("quality_flag") == "failed"

    # Selection margin is reported against the best unselected candidate.
    other_scores = [
        float(candidate.get("quality_score", -999))
        for candidate in candidates
        if candidate is not selected
    ]
    second_best_score = (
        max(other_scores)
        if other_scores
        else None
    )
    selection_margin = (
        float(selected.get("quality_score", -999))
        - second_best_score
        if second_best_score is not None
        else None
    )

    clean_success = bool(
        clean_text
        and selected.get("quality_flag") != "failed"
        and float(
            selected.get("quality_score", -999)
        ) >= MIN_SCORE_ACCEPTABLE
    )

    selected_meta = selected.get("meta", {})

    result = {
        **base,
        "clean_success": clean_success,
        "selected_method": selected.get(
            "candidate_method"
        ),
        "extraction_engine": selected.get(
            "candidate_method"
        ),
        "mineru_used": selected.get(
            "candidate_method"
        ) == "pdf_mineru",
        "quality_score": selected.get(
            "quality_score"
        ),
        "quality_flag": selected.get(
            "quality_flag"
        ),
        "quality_reasons": selected.get(
            "quality_reasons"
        ),
        "needs_manual_review": needs_manual_review,
        "n_chars_raw": selected.get(
            "n_chars_raw"
        ),
        "n_chars_clean": selected.get(
            "n_chars_clean"
        ),
        "n_pages": selected.get("n_pages"),
        "detected_language": selected.get(
            "detected_language"
        ),
        "language_score": selected.get(
            "language_score"
        ),
        "replacement_ratio": selected.get(
            "replacement_ratio"
        ),
        "mojibake_ratio": selected.get(
            "mojibake_ratio"
        ),
        "private_use_ratio": selected.get(
            "private_use_ratio"
        ),
        "repeated_character_ratio": selected.get(
            "repeated_character_ratio"
        ),
        "duplicate_line_ratio": selected.get(
            "duplicate_line_ratio"
        ),
        "one_char_token_ratio": selected.get(
            "one_char_token_ratio"
        ),
        "very_long_token_ratio": selected.get(
            "very_long_token_ratio"
        ),
        "no_vowel_token_ratio": selected.get(
            "no_vowel_token_ratio"
        ),
        "candidate_count": len(candidates),
        "candidate_methods": "|".join(
            candidate["candidate_method"]
            for candidate in candidates
        ),
        "candidate_scores": score_listing,
        "second_best_score": second_best_score,
        "selection_margin": selection_margin,
        "selection_confidence": (
            "single_candidate"
            if selection_margin is None
            else "high"
            if selection_margin >= 5
            else "medium"
            if selection_margin >= 1
            else "low"
        ),
        "selection_reason": selection_reason,
        "selected_encoding": selected_meta.get(
            "encoding",
            "",
        ),
        "selected_encoding_source": (
            selected_meta.get(
                "encoding_source",
                "",
            )
        ),
        "selected_ftfy_applied": selected_meta.get(
            "ftfy_applied",
            False,
        ),
        "candidate_errors": " | ".join(errors),
        "error": (
            ""
            if clean_success
            else "selected_candidate_below_quality_threshold"
        ),
    }

    return result, candidate_rows


target_rows = work_manifest.to_dict("records")
results: List[Dict[str, Any]] = []
candidate_results: List[Dict[str, Any]] = []

for index, row in enumerate(
    tqdm(
        target_rows,
        desc="Cleaning DB-COMP files v3",
    ),
    start=1,
):
    result, candidate_rows = process_one(row)
    results.append(result)
    candidate_results.extend(candidate_rows)

    if SAVE_EVERY and index % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(
            CLEAN_FILE_MANIFEST_PATH,
            index=False,
            encoding="utf-8",
        )

clean_file_manifest = pd.DataFrame(results)
candidate_manifest = pd.DataFrame(candidate_results)

essential_cols = [
    "document_id",
    "dbcomp_document_id",
    "case_number",
    "title_for_processing",
    "source_url_for_processing",
    "raw_file_path",
    "raw_file_type",
    "file_format",
    "processing_eligible",
    "processing_skip_reason",
    "cleaning_target",
    "clean_success",
    "extraction_version",
    "quality_flag",
    "quality_score",
    "quality_reasons",
    "needs_manual_review",
    "n_chars_raw",
    "n_chars_clean",
    "n_pages",
    "selected_method",
    "extraction_engine",
    "mineru_used",
    "candidate_count",
    "candidate_methods",
    "candidate_scores",
    "second_best_score",
    "selection_margin",
    "selection_confidence",
    "selection_reason",
    "candidate_errors",
    "detected_language",
    "language_score",
    "replacement_ratio",
    "mojibake_ratio",
    "private_use_ratio",
    "repeated_character_ratio",
    "duplicate_line_ratio",
    "one_char_token_ratio",
    "very_long_token_ratio",
    "no_vowel_token_ratio",
    "selected_encoding",
    "selected_encoding_source",
    "selected_ftfy_applied",
    "clean_text_path",
    "readable_text_path",
    "markdown_path",
    "processed_at",
    "error",
]
essential_cols = [
    column
    for column in essential_cols
    if column in clean_file_manifest.columns
]
clean_file_manifest = (
    clean_file_manifest[essential_cols].copy()
)

clean_file_manifest.to_csv(
    CLEAN_FILE_MANIFEST_PATH,
    index=False,
    encoding="utf-8",
)
try:
    clean_file_manifest.to_excel(
        CLEAN_FILE_MANIFEST_XLSX_PATH,
        index=False,
    )
except Exception as exc:
    print(
        f"Warning: could not write Excel manifest: {exc}"
    )

if not candidate_manifest.empty:
    candidate_manifest.to_csv(
        CANDIDATE_MANIFEST_PATH,
        index=False,
        encoding="utf-8",
    )
elif not CANDIDATE_MANIFEST_PATH.exists():
    pd.DataFrame(
        columns=[
            "document_id",
            "candidate_method",
            "quality_score",
            "quality_flag",
        ]
    ).to_csv(
        CANDIDATE_MANIFEST_PATH,
        index=False,
        encoding="utf-8",
    )

failed = clean_file_manifest[
    ~clean_file_manifest[
        "clean_success"
    ].fillna(False)
].copy()
failed.to_csv(
    FAILED_FILE_MANIFEST_PATH,
    index=False,
    encoding="utf-8",
)

print("Clean manifest CSV:", CLEAN_FILE_MANIFEST_PATH)
print("Clean manifest XLSX:", CLEAN_FILE_MANIFEST_XLSX_PATH)
print("Candidate manifest:", CANDIDATE_MANIFEST_PATH)
print("Failed manifest:", FAILED_FILE_MANIFEST_PATH)
print("Rows:", len(clean_file_manifest))
print(
    "Clean success:",
    int(
        clean_file_manifest[
            "clean_success"
        ].fillna(False).sum()
    ),
)
print("Failed/skipped:", len(failed))
display(clean_file_manifest.head(20))


Cleaning DB-COMP files v3:   0%|          | 0/833 [00:00<?, ?it/s]

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library err

,document_id,dbcomp_document_id,case_number,title_for_processing,source_url_for_processing,raw_file_path,raw_file_type,file_format,processing_eligible,processing_skip_reason,...,very_long_token_ratio,no_vowel_token_ratio,selected_encoding,selected_encoding_source,selected_ftfy_applied,clean_text_path,readable_text_path,markdown_path,processed_at,error
0,1061,1061,93,,https://db-comp.eu/document_1061.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.000000,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:13:25,
1,1064,1064,399904,,https://db-comp.eu/document_1064.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.001054,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:13:26,
2,1065,1065,40481,,https://db-comp.eu/document_1065.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.001734,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:13:26,
3,1066,1066,40360,,https://db-comp.eu/document_1066.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.000000,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:13:55,
4,1067,1067,40291,,https://db-comp.eu/document_1067.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.000615,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:13:55,
5,1068,1068,40208,,https://db-comp.eu/document_1068.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.001015,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:13:56,
6,1069,1069,40169,,https://db-comp.eu/document_1069.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.000971,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:14:40,
7,1072,1072,40113,,https://db-comp.eu/document_1072.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.001279,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:14:40,
8,1073,1073,40105,,https://db-comp.eu/document_1073.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.002330,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:15:06,
9,1074,1074,40098,,https://db-comp.eu/document_1074.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf,True,,...,0.000000,0.001462,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-07-16T18:15:06,


## 7. Validation summary

In [22]:

if (
    CLEAN_FILE_MANIFEST_PATH.exists()
    and CLEAN_FILE_MANIFEST_PATH.stat().st_size > 0
):
    manifest = pd.read_csv(
        CLEAN_FILE_MANIFEST_PATH,
        low_memory=False,
    )

    print("Rows:", len(manifest))
    print(
        "Successful:",
        int(
            manifest["clean_success"]
            .fillna(False)
            .sum()
        ),
    )
    print(
        "Needs review:",
        int(
            manifest.get(
                "needs_manual_review",
                pd.Series(False, index=manifest.index),
            )
            .fillna(False)
            .sum()
        ),
    )

    display(
        manifest[
            [
                column
                for column in [
                    "file_format",
                    "selected_method",
                    "quality_flag",
                    "mineru_used",
                ]
                if column in manifest.columns
            ]
        ].value_counts(dropna=False).reset_index(
            name="n"
        )
    )

    review_cols = [
        column
        for column in [
            "document_id",
            "case_number",
            "file_format",
            "selected_method",
            "quality_score",
            "quality_flag",
            "quality_reasons",
            "needs_manual_review",
            "candidate_scores",
            "selection_reason",
            "clean_text_path",
        ]
        if column in manifest.columns
    ]

    display(
        manifest.sort_values(
            [
                "needs_manual_review",
                "quality_score",
            ],
            ascending=[False, True],
            na_position="first",
        )[review_cols].head(100)
    )
else:
    print("Run the processing cell first.")

if (
    CANDIDATE_MANIFEST_PATH.exists()
    and CANDIDATE_MANIFEST_PATH.stat().st_size > 0
):
    try:
        candidates = pd.read_csv(
            CANDIDATE_MANIFEST_PATH,
            low_memory=False,
        )
    except pd.errors.EmptyDataError:
        candidates = pd.DataFrame()

    if not candidates.empty:
        selected_lookup = (
            manifest[
                [
                    "document_id",
                    "selected_method",
                    "candidate_count",
                    "quality_score",
                ]
            ]
            .copy()
        )
        candidate_counts = (
            candidates.groupby(
                candidates["document_id"].astype(str)
            )
            .size()
        )

        selected_lookup["actual_candidate_count"] = (
            selected_lookup["document_id"]
            .astype(str)
            .map(candidate_counts)
        )

        selected_scores = (
            candidates.groupby(
                candidates["document_id"].astype(str)
            )["quality_score"]
            .max()
        )
        selected_lookup["max_candidate_score"] = (
            selected_lookup["document_id"]
            .astype(str)
            .map(selected_scores)
        )

        # MinerU gain policy means selected score may intentionally be below
        # the maximum if OCR did not beat native by the required margin.
        selected_lookup["candidate_count_matches"] = (
            selected_lookup["candidate_count"]
            == selected_lookup[
                "actual_candidate_count"
            ]
        )

        display(
            selected_lookup[
                ~selected_lookup[
                    "candidate_count_matches"
                ].fillna(False)
            ]
        )

        comparison = (
            candidates.groupby(
                [
                    "file_format",
                    "candidate_method",
                    "quality_flag",
                ],
                dropna=False,
            )
            .agg(
                n=("document_id", "size"),
                mean_score=("quality_score", "mean"),
                median_score=("quality_score", "median"),
                mean_chars=("n_chars_clean", "mean"),
            )
            .reset_index()
            .sort_values(
                ["file_format", "mean_score"],
                ascending=[True, False],
            )
        )
        display(comparison)


Rows: 833
Successful: 833
Needs review: 68


,file_format,selected_method,quality_flag,mineru_used,n
0,pdf,pdf_pymupdf_text_sorted,ok,False,408
1,pdf,pdf_pymupdf_blocks_sorted,ok,False,163
2,pdf,pdf_mineru,ok,True,142
3,pdf,pdf_pdftotext_layout,ok,False,47
4,pdf,pdf_pymupdf_text_sorted,fishy,False,38
5,pdf,pdf_pymupdf_blocks_sorted,fishy,False,29
6,pdf,pdf_pdftotext_layout,fishy,False,6


,document_id,case_number,file_format,selected_method,quality_score,quality_flag,quality_reasons,needs_manual_review,candidate_scores,selection_reason,clean_text_path
231,1368,36282,pdf,pdf_pdftotext_layout,41.653,fishy,short_pdf_text|repeated_character_garbage,True,pdf_mineru=43.768; pdf_pdftotext_layout=41.653...,mineru_gain_2.115_below_threshold,/home/edik/projects/eccjeu/data/processed/com_...
811,2696,40577,pdf,pdf_pymupdf_blocks_sorted,65.558,fishy,repeated_character_garbage,True,pdf_mineru=67.905; pdf_pymupdf_blocks_sorted=6...,mineru_gain_2.347_below_threshold,/home/edik/projects/eccjeu/data/processed/com_...
169,1296,37784,pdf,pdf_pymupdf_text_sorted,65.774,fishy,repeated_character_garbage,True,pdf_mineru=68.132; pdf_pymupdf_text_sorted=65....,mineru_gain_2.358_below_threshold,/home/edik/projects/eccjeu/data/processed/com_...
634,1883,40220,pdf,pdf_pymupdf_blocks_sorted,65.835,fishy,repeated_character_garbage,True,pdf_mineru=68.310; pdf_pymupdf_blocks_sorted=6...,mineru_gain_2.475_below_threshold,/home/edik/projects/eccjeu/data/processed/com_...
804,2684,40632,pdf,pdf_pymupdf_text_sorted,65.900,fishy,repeated_character_garbage,True,pdf_mineru=67.861; pdf_pymupdf_text_sorted=65....,mineru_gain_1.961_below_threshold,/home/edik/projects/eccjeu/data/processed/com_...
...,...,...,...,...,...,...,...,...,...,...,...
47,1137,39699,pdf,pdf_pdftotext_layout,46.822,ok,NaN,False,pdf_pdftotext_layout=46.822; pdf_pymupdf_text_...,best_available_candidate,/home/edik/projects/eccjeu/data/processed/com_...
137,1258,38229,pdf,pdf_pymupdf_text_sorted,46.867,ok,NaN,False,pdf_pymupdf_text_sorted=46.867; pdf_pymupdf_bl...,best_available_candidate,/home/edik/projects/eccjeu/data/processed/com_...
203,1337,36844,pdf,pdf_pdftotext_layout,46.875,ok,NaN,False,pdf_pdftotext_layout=46.875; pdf_pymupdf_block...,best_available_candidate,/home/edik/projects/eccjeu/data/processed/com_...
48,1138,39687,pdf,pdf_pdftotext_layout,46.897,ok,NaN,False,pdf_pdftotext_layout=46.897; pdf_pymupdf_block...,best_available_candidate,/home/edik/projects/eccjeu/data/processed/com_...


,document_id,selected_method,candidate_count,quality_score,actual_candidate_count,max_candidate_score,candidate_count_matches


,file_format,candidate_method,quality_flag,n,mean_score,median_score,mean_chars
1,pdf,pdf_mineru,ok,209,68.276689,68.5040,214476.952153
10,pdf,pdf_pymupdf_text_sorted,ok,617,65.556049,68.8890,73604.564019
7,pdf,pdf_pymupdf_blocks_sorted,ok,614,65.283995,68.8415,74634.763844
4,pdf,pdf_pdftotext_layout,ok,614,65.219523,68.8460,73802.131922
6,pdf,pdf_pymupdf_blocks_sorted,fishy,196,64.224638,65.4550,261130.760204
9,pdf,pdf_pymupdf_text_sorted,fishy,197,64.157883,65.4500,260724.974619
3,pdf,pdf_pdftotext_layout,fishy,197,63.971756,65.2360,259890.994924
0,pdf,pdf_mineru,fishy,1,43.768000,43.7680,885.000000
5,pdf,pdf_pymupdf_blocks_sorted,failed,23,17.949609,15.7490,515.652174
8,pdf,pdf_pymupdf_text_sorted,failed,19,15.241474,15.3770,201.842105


## 8. Inspect one processed document

In [23]:

DBCOMP_DOCUMENT_ID_TO_VIEW = "1368"
# Example:
# DBCOMP_DOCUMENT_ID_TO_VIEW = "9000"

if DBCOMP_DOCUMENT_ID_TO_VIEW:
    manifest = pd.read_csv(
        CLEAN_FILE_MANIFEST_PATH,
        low_memory=False,
    )
    hit = manifest[
        manifest["dbcomp_document_id"]
        .astype(str)
        .eq(str(DBCOMP_DOCUMENT_ID_TO_VIEW))
    ]

    if hit.empty:
        print("Document ID not found.")
    else:
        record = hit.iloc[-1].to_dict()

        keys = [
            "document_id",
            "dbcomp_document_id",
            "case_number",
            "selected_method",
            "quality_score",
            "quality_flag",
            "quality_reasons",
            "candidate_scores",
            "selection_reason",
            "clean_text_path",
            "readable_text_path",
        ]
        print(
            json.dumps(
                {
                    key: record.get(key)
                    for key in keys
                },
                indent=2,
                ensure_ascii=False,
            )
        )

        print(
            "\n--- CANONICAL REGEX/LLM VERSION ---\n"
        )
        print(
            Path(
                record["clean_text_path"]
            ).read_text(
                encoding="utf-8",
                errors="replace",
            )[:7000]
        )

        print("\n--- READABLE VERSION ---\n")
        print(
            Path(
                record["readable_text_path"]
            ).read_text(
                encoding="utf-8",
                errors="replace",
            )[:7000]
        )

        if (
            CANDIDATE_MANIFEST_PATH.exists()
            and CANDIDATE_MANIFEST_PATH.stat().st_size > 0
        ):
            candidates = pd.read_csv(
                CANDIDATE_MANIFEST_PATH,
                low_memory=False,
            )
            display(
                candidates[
                    candidates["document_id"]
                    .astype(str)
                    .eq(str(record["document_id"]))
                ].sort_values(
                    "quality_score",
                    ascending=False,
                )
            )
else:
    print(
        "Set DBCOMP_DOCUMENT_ID_TO_VIEW to inspect a document."
    )


{
  "document_id": 1368,
  "dbcomp_document_id": 1368,
  "case_number": "36282",
  "selected_method": "pdf_pdftotext_layout",
  "quality_score": 41.653,
  "quality_flag": "fishy",
  "quality_reasons": "short_pdf_text|repeated_character_garbage",
  "candidate_scores": "pdf_mineru=43.768; pdf_pdftotext_layout=41.653; pdf_pymupdf_text_sorted=41.640; pdf_pymupdf_blocks_sorted=41.606",
  "selection_reason": "mineru_gain_2.115_below_threshold",
  "clean_text_path": "/home/edik/projects/eccjeu/data/processed/com_db_comp/text/1368.txt",
  "readable_text_path": "/home/edik/projects/eccjeu/data/processed/com_db_comp/text/1368__readable.txt"
}

--- CANONICAL REGEX/LLM VERSION ---

EUROPEAN COMMISSION Y.f •J ~

.;:r """'~U ·- - - - -'

Brussels, U~ -u:t" 't' SG(97) DI 14 I 9

REGISTERED WITH ACKNOWLEDGEMENT ~r RECEIPT

Subject: Case Nci IV/36.282 - VSA Notification

Further to the publieation of a Notice in the Official Journal concerning the above case (OJ No C 185, 18.6.1997, p. 4), I am pleased

,document_id,dbcomp_document_id,case_number,raw_file_path,file_format,candidate_method,quality_score,quality_flag,quality_reasons,n_chars_raw,...,language_score,replacement_ratio,mojibake_ratio,private_use_ratio,repeated_character_ratio,duplicate_line_ratio,one_char_token_ratio,very_long_token_ratio,no_vowel_token_ratio,meta_json
777,1368,1368,36282,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf_mineru,43.768,fishy,short_pdf_text|many_very_long_tokens,884,...,1.0,0.0,0.0,0.0,0.000000,0.0,0.061538,0.015385,0.000000,"{""method"": ""pdf_mineru"", ""n_pages"": 1}"
776,1368,1368,36282,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf_pdftotext_layout,41.653,fishy,short_pdf_text|repeated_character_garbage,1808,...,1.0,0.0,0.0,0.0,0.016935,0.0,0.157635,0.000000,0.004926,"{""method"": ""pdf_pdftotext_layout"", ""n_pages"": 1}"
774,1368,1368,36282,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf_pymupdf_text_sorted,41.640,fishy,short_pdf_text|repeated_character_garbage,1682,...,1.0,0.0,0.0,0.0,0.016920,0.0,0.160976,0.000000,0.004878,"{""method"": ""pdf_pymupdf_text_sorted"", ""n_pages..."
775,1368,1368,36282,/home/edik/projects/eccjeu/data/raw/com_db_com...,pdf,pdf_pymupdf_blocks_sorted,41.606,fishy,short_pdf_text|repeated_character_garbage,1205,...,1.0,0.0,0.0,0.0,0.016835,0.0,0.160976,0.000000,0.004878,"{""method"": ""pdf_pymupdf_blocks_sorted"", ""n_pag..."
